# Accessing NCBI Entrez Utilities (EUtils) with BioServices

This notebook demonstrates how to use BioServices to access the NCBI Entrez Programming
Utilities (E-utilities), which provide a stable interface to the Entrez query and database
system at NCBI.

The examples are inspired by the NCBI E-utilities cookbook:
https://www.ncbi.nlm.nih.gov/books/NBK25498/

- [Setup](#setup)
- [ESearch and ESummary](#esearch)
- [EPost and EFetch](#epost)
- [Application 1: GI numbers to accession numbers](#app1)
- [Application 2: Accession numbers to FASTA data](#app2)
- [Application 3: Retrieving large datasets in batches](#app3)
- [Application 4: ELink – finding related records](#app4)
- [Additional utilities](#extra)

**References**: https://www.ncbi.nlm.nih.gov/books/NBK25499/

## <a name="setup"></a> Setup

You should provide your email address when creating the `EUtils` instance.  NCBI uses
it to contact you before blocking access if you exceed the rate limit (3 requests/s).
You can provide it in three ways:

1. Pass it as a parameter: `EUtils(email="your@email.com")`
2. Set it afterwards: `e.email = "your@email.com"`
3. Add it to `~/.config/bioservices/bioservices.cfg` under `[user] / email = …`

In [ ]:
from bioservices import EUtils
e = EUtils(email="test@example.com")

## <a name="esearch"></a> ESearch and ESummary / EFetch

**ESearch** searches an Entrez database for a query term and returns a list of matching
UIDs (or stores them on NCBI's History server when `usehistory='y'` is set).  
**ESummary** returns short document summaries for a set of UIDs.  
**EFetch** returns full records in a variety of formats (FASTA, GenBank, etc.).

The example below downloads PubMed records indexed in MeSH for both *asthma* **and**
*leukotrienes* that were published in 2009.

In [ ]:
# Search PubMed for records about asthma and leukotrienes published in 2009.
# usehistory='y' stores the result on NCBI's History server so we can re-use
# the WebEnv/query_key in subsequent calls without re-sending all IDs.
query = 'asthma[mesh]+AND+leukotrienes[mesh]+AND+2009[pdat]'
db = 'pubmed'
results = e.ESearch(db, query, usehistory='y')
print("Total hits :", results['count'])
print("WebEnv     :", results['webenv'])
print("Query key  :", results['querykey'])
print("First IDs  :", results['idlist'][:5])

In [ ]:
# Retrieve document summaries via the History server (no need to repeat the IDs)
summaries = e.ESummary(db, query_key=results['querykey'], WebEnv=results['webenv'])
print("Number of summaries returned:", len(summaries))

## <a name="epost"></a> EPost and EFetch

**EPost** uploads a list of UIDs to the History server and returns a `WebEnv` /
`QueryKey` pair that can be used in subsequent `ESummary` or `EFetch` calls.
This is more efficient than embedding thousands of IDs in a URL.

- **Input**: comma-delimited list of Entrez UIDs (PMID, GI number, Gene ID, …)
- **ESummary output**: XML document summaries
- **EFetch output**: formatted records (FASTA, GenBank, abstracts, …)

In [ ]:
db = 'protein'
id_list = '194680922,50978626,28558982,9507199,6678417'

# Upload the IDs to the History server
post_result = e.EPost(db, id_list)
print(post_result)

In [ ]:
# Retrieve FASTA sequences for all posted IDs
fasta = e.EFetch(
    db, id_list,
    query_key=post_result['QueryKey'],
    WebEnv=post_result['WebEnv'],
    retmode='text',
    rettype='fasta',
)
# fasta may be bytes or str depending on the server response
if isinstance(fasta, bytes):
    fasta = fasta.decode()
print(fasta[:600])

## <a name="app1"></a> Application 1: Converting GI numbers to accession numbers

- **Goal**: Given a list of nucleotide GI numbers, retrieve the corresponding accession
  numbers.
- **Solution**: Use `EFetch` with `rettype='acc'`.
- **Input**: comma-delimited list of GI numbers
- **Output**: list of accession numbers (same order as the input)

In [ ]:
gi_list = '24475906,224465210,50978625,9507198'
output = e.EFetch('nucleotide', gi_list, rettype='acc')
if isinstance(output, bytes):
    output = output.decode()
accessions = output.split()
print("Accession numbers:", accessions)

## <a name="app2"></a> Application 2: Converting accession numbers to FASTA data

- **Goal**: Given a list of nucleotide accession numbers, retrieve their sequences in
  FASTA format.
- **Solution**: Build an ESearch `term` by joining accessions with `+OR+` (each
  followed by `[accn]`), then use `EFetch` to retrieve the FASTA data.
- **Input**: comma-delimited list of accession numbers
- **Output**: FASTA sequences

In [ ]:
acc_list = 'NM_009417,NM_000547,NM_001003009,NM_019353'

# Build a query string where each accession is searched in the [accn] field
term = '+OR+'.join(acc + '[accn]' for acc in acc_list.split(','))
search = e.ESearch(db='nucleotide', term=term, usehistory='y')
print("Found", search['count'], "records")
print("IDs  :", search['idlist'])

In [ ]:
# Fetch FASTA data using the History server
fasta = e.EFetch(
    'nucleotide',
    id=','.join(search['idlist']),
    query_key=search['querykey'],
    WebEnv=search['webenv'],
    rettype='fasta',
    retmode='text',
)
if isinstance(fasta, bytes):
    fasta = fasta.decode()
print(fasta[:600])

## <a name="app3"></a> Application 3: Retrieving large datasets in batches

- **Goal**: Download all chimpanzee mRNA sequences in FASTA format (>50,000 records).
- **Solution**: Use `ESearch` to post the query on the History server, then loop over
  `EFetch` calls in batches of `retmax` records each.
- **Note**: The full download would take a very long time.  In this example we cap the
  number of batches at 2 (i.e. 1,000 sequences) for illustration purposes.

In [ ]:
query = 'chimpanzee[orgn]+AND+biomol+mrna[prop]'
search = e.ESearch(db='nucleotide', term=query, usehistory='y')
total = int(search['count'])
print("Total chimpanzee mRNA records:", total)

In [ ]:
import math
import tempfile

retmax = 500                         # records per batch
max_batches = 2                      # limit for this demonstration
n_batches = min(max_batches, math.ceil(total / retmax))

with tempfile.NamedTemporaryFile(mode='w', suffix='.fna', delete=False) as fh:
    output_file = fh.name
    for i in range(n_batches):
        data = e.EFetch(
            'nucleotide',
            WebEnv=search['webenv'],
            query_key=search['querykey'],
            retstart=i * retmax,
            retmax=retmax,
            retmode='text',
            rettype='fasta',
        )
        if isinstance(data, bytes):
            data = data.decode()
        fh.write(data)
        print(f"Batch {i + 1}/{n_batches} written ({retmax} records)")

print(f"\nData saved to {output_file}")

## <a name="app4"></a> Application 4: ELink – finding related records

**ELink** lets you follow links between records in different Entrez databases (or within
the same database).  Common use-cases:

- Find SNPs (dbSNP) linked to a gene
- Find the protein record corresponding to a nucleotide
- Find related PubMed articles

The example below links nucleotide records to their corresponding protein sequences.

In [ ]:
# Link two nucleotide GI numbers to their protein translations
result_xml = e.ELink(dbfrom='nucleotide', db='protein', id='48819,7140345')
if isinstance(result_xml, bytes):
    result_xml = result_xml.decode()
print(result_xml[:800])

In [ ]:
# Link nucleotide records to their taxonomy entries
result_xml = e.ELink(dbfrom='nuccore', db='taxonomy', id='21614549,219152114')
if isinstance(result_xml, bytes):
    result_xml = result_xml.decode()
print(result_xml[:800])

In [ ]:
# Find related PubMed articles for a given PMID using 'neighbor_score' command
result_xml = e.ELink('pubmed', id='20210808', cmd='neighbor_score')
if isinstance(result_xml, bytes):
    result_xml = result_xml.decode()
print(result_xml[:800])

## <a name="extra"></a> Additional utilities

### EInfo – database metadata

`EInfo` returns general information about an Entrez database (field names, link names,
total number of records, …).  When called without a database argument it returns the
list of all available databases.

In [ ]:
# List all available Entrez databases
print("Available databases:", e.databases[:10], "...")

In [ ]:
# Get metadata for the PubMed database
# EInfo returns a list; the first element contains the database stats
info = e.EInfo('pubmed')
print("PubMed record count:", info[0]['count'])
print("Last update        :", info[0]['lastupdate'])

### EGQuery – global query across all databases

In [ ]:
# Search for 'asthma' across all Entrez databases at once.
# EGQuery returns a parsed XML result; ResultItem contains per-database counts.
gq = e.EGQuery('asthma')
# gq is an EUtilsParser object; index into ResultItem for the database breakdown
items = gq['ResultItem']
hits = [(item['DbName'], int(item['Count'])) for item in items if item['Status'] == 'Ok']
hits.sort(key=lambda x: x[1], reverse=True)
for db_name, count in hits[:10]:
    print(f"{db_name:20s}  {count:>10,}")

### ESpell – spelling suggestions

In [ ]:
# Ask NCBI to suggest a corrected spelling for a mis-typed query
spell = e.ESpell(db='pubmed', term='biosservices')
print("Corrected query:", spell['eSpellResult']['CorrectedQuery'])